# 📊 NeMo Fraud Detection: LoRA-Adapter Evaluierung

Dieses Notebook führt eine lokale Evaluierung des feinabgestimmten LoRA-Adapters (**Llama-3.1-8B-Instruct**) für die Betrugserkennung durch.

### Funktionsumfang:
1. **Modell-Initialisierung:** Lädt das Basismodell und verknüpft den trainierten LoRA-Adapter.
2. **Inferenz:** Generiert Antworten lokal via PyTorch/Transformers auf Basis der Validierungstranskripte.
3. **Parsing & Validierung:** Mappt Ground-Truth-Labels und vergleicht sie robust mit der Modellantwort.
4. **Metriken & Tracking:** Berechnet die Accuracy und loggt die Ergebnisse an Weights & Biases (`wandb`).

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import wandb

# Konfiguration der Pfade und Modelle
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
ADAPTER_PATH = "/data/nemo-fraud-detection/notebooks/04_FineTuning/evaluation/fraud_detection_qlora"
VAL_FILE = "/data/nemo-fraud-detection/notebooks/02_Data_Curation/sft/validation.jsonl"

print("✅ Bibliotheken und Konfigurationen geladen.")

### 1. Durchführung der LoRA-Evaluierung

In [ ]:
def run_lora_evaluation():
    print("🚀 Lade Basis-Modell und LoRA-Adapter...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    model.eval()

    wandb.init(project="fraud-detection", name="lora-evaluation")
    correct, total = 0, 0

    with open(VAL_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            input_text = f"### Instruction:\n{data['input']}\n\n### Response:\n"
            
            inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=5)
            
            response = tokenizer.decode(outputs[0], skip_special_tokens=True).split("### Response:\n")[-1].strip().lower()
            
            true_label = "fraud" if "betrug" in data['output'].lower() else "legitimate"
            model_answer = "fraud" if "fraud" in response else "legitimate"
            
            is_correct = (true_label == model_answer)
            if is_correct:
                correct += 1
            total += 1
            print(f"Erwartet: {true_label} | Erkannt: {model_answer} | Korrekt: {is_correct}")

    accuracy = (correct / total) * 100 if total > 0 else 0
    print(f"\n🎯 Lora-Evaluierungs-Genauigkeit: {accuracy:.2f}%")
    wandb.log({"eval/accuracy": accuracy})
    wandb.finish()

# Start der LoRA-Evaluierung
run_lora_evaluation()